# 00. Data audit (step 0.2)

Reproduces every figure in `docs/data_audit.md`. Run it top to bottom from the repository
root or from `notebooks/`:

```
uv sync --frozen --group notebooks
KAGGLE_API_TOKEN=... uv run --frozen jupyter nbconvert --to notebook --execute --inplace notebooks/00_data_audit.ipynb
```

What it does:

1. Downloads the raw files from Kaggle (only if missing) and checks their SHA-256.
2. Builds the inventory and the data dictionary, with availability at listing time.
3. Checks leakage, labels and their delay, data quality, volumes over time and the
   `addr_state` / zip code proxy question.

Every table it produces is written to `reports/data_audit/`. Raw data lives in `data/`
and is never committed.

In [1]:
import hashlib
import json
import platform
import re
import subprocess
from pathlib import Path

import matplotlib

matplotlib.use("Agg")
import matplotlib.pyplot as plt
import numpy as np
import polars as pl
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import log_loss, roc_auc_score
from sklearn.preprocessing import OneHotEncoder

# Built-in defaults, not the machine's matplotlibrc, so figures look the same everywhere (E-024).
matplotlib.rcdefaults()
pl.Config.set_tbl_rows(80)
pl.Config.set_tbl_cols(20)
pl.Config.set_tbl_width_chars(200)
pl.Config.set_fmt_str_lengths(60)

ROOT = Path.cwd() if (Path.cwd() / "docs").exists() else Path.cwd().parent
RAW = ROOT / "data" / "raw"
CACHE = ROOT / "data" / "interim"
OUT = ROOT / "reports" / "data_audit"
for p in (RAW, CACHE, OUT):
    p.mkdir(parents=True, exist_ok=True)

print("python", platform.python_version(), "| polars", pl.__version__)

python 3.12.13 | polars 2.1.0


## 1. Download and verify

Two Kaggle datasets:

| Dataset | Used for |
|---|---|
| `wordsforthewise/lending-club`, file `accepted_2007_to_2018Q4.csv.gz` | The loans |
| `jonchan2003/lending-club-data-dictionary` | Column descriptions. Its `Notes` sheet is Lending Club's *Browse Notes* dictionary: the fields an investor saw on a listed loan. It is the evidence for "available at listing" |

The expected hashes were recorded at the first download (2026-11-21). If Kaggle ever
serves different bytes, the notebook stops here instead of silently auditing other data.

In [2]:
SOURCES = {
    "accepted_2007_to_2018Q4.csv.gz": {
        "dataset": "wordsforthewise/lending-club",
        "sha256": "55c16f75120f897683f02e7aabcf080d0e4a20c4832feb1d592cfa941bd62a2d",
    },
    "Lending Club Data Dictionary Approved.csv": {
        "dataset": "jonchan2003/lending-club-data-dictionary",
        "sha256": "c0fecbb9129fe2d685ccd98a9cf4e2580c032b8d642171bd57aef3da9c53d68d",
    },
    "Lending Club Data Dictionary Notes.csv": {
        "dataset": "jonchan2003/lending-club-data-dictionary",
        "sha256": "fc7fd3763d73cf15bce13cdecf33015a059000ffe92000241c23b72610003cf8",
    },
}


def sha256(path: Path) -> str:
    h = hashlib.sha256()
    with path.open("rb") as f:
        for chunk in iter(lambda: f.read(1 << 20), b""):
            h.update(chunk)
    return h.hexdigest()


def kaggle_download(dataset: str, file: str | None, dest: Path) -> None:
    cmd = ["kaggle", "datasets", "download", dataset, "-p", str(dest)]
    cmd += ["-f", file] if file else ["--unzip"]
    subprocess.run(cmd, check=True, capture_output=True)


for name, src in SOURCES.items():
    path = RAW / name
    if not path.exists():
        single_file = name if name.endswith(".gz") else None
        kaggle_download(src["dataset"], single_file, RAW)
    got = sha256(path)
    assert got == src["sha256"], f"{name}: hash {got} does not match the recorded one"
    print(f"OK  {name}  {path.stat().st_size:,} bytes  sha256 {got[:16]}...")

OK  accepted_2007_to_2018Q4.csv.gz  392,582,231 bytes  sha256 55c16f75120f8976...
OK  Lending Club Data Dictionary Approved.csv  13,780 bytes  sha256 c0fecbb9129fe2d6...
OK  Lending Club Data Dictionary Notes.csv  9,960 bytes  sha256 fc7fd3763d73cf15...


In [3]:
# Licence and description, straight from Kaggle's metadata.
licences = {}
for dataset in sorted({s["dataset"] for s in SOURCES.values()}):
    meta_dir = CACHE / ("meta_" + dataset.replace("/", "_"))
    meta_file = meta_dir / "dataset-metadata.json"
    if not meta_file.exists():
        subprocess.run(
            ["kaggle", "datasets", "metadata", dataset, "-p", str(meta_dir)],
            check=True,
            capture_output=True,
        )
    info = json.loads(meta_file.read_text())["info"]
    licences[dataset] = [lic["name"] for lic in info["licenses"]]
print(licences)

{'jonchan2003/lending-club-data-dictionary': ['unknown'], 'wordsforthewise/lending-club': ['CC0-1.0']}


Load the CSV once and cache it as Parquet (about 50 s for the CSV, under 5 s for Parquet).
Every column type is inferred from the whole file, not from the first rows.

In [4]:
PARQUET = CACHE / "accepted.parquet"
if not PARQUET.exists():
    pl.read_csv(
        RAW / "accepted_2007_to_2018Q4.csv.gz", infer_schema_length=None, low_memory=False
    ).write_parquet(PARQUET)
raw = pl.read_parquet(PARQUET)
print("raw shape", raw.shape)

raw shape (2260701, 151)


## 2. Inventory

The file ends with summary rows that Lending Club appended to its quarterly exports
("Total amount funded in policy code ..."). They have text in `id` and nulls everywhere
else, so they are detected by a missing `loan_amnt`.

In [5]:
footer = raw.filter(pl.col("loan_amnt").is_null())
print("footer rows:", footer.height)
print(footer.select(pl.col("id").str.slice(0, 45)).unique())

df = raw.filter(pl.col("loan_amnt").is_not_null())
RAW_COLS, RAW_SCHEMA, RAW_ROWS = raw.columns, raw.schema, raw.height
del raw  # keep memory low: the full table is held once, as `df`


def month(col: str) -> pl.Expr:
    """Parse Lending Club month strings such as 'Dec-2015' into the first day of the month."""
    return ("01-" + pl.col(col)).str.strptime(pl.Date, "%d-%b-%Y", strict=False)


DATE_COLS = [
    "issue_d",
    "last_pymnt_d",
    "next_pymnt_d",
    "last_credit_pull_d",
    "earliest_cr_line",
    "sec_app_earliest_cr_line",
    "hardship_start_date",
    "hardship_end_date",
    "payment_plan_start_date",
    "debt_settlement_flag_date",
    "settlement_date",
]
df = df.with_columns([month(c).alias(c + "_dt") for c in DATE_COLS])
df = df.with_columns(
    issue_year=pl.col("issue_d_dt").dt.year(),
    term_m=pl.col("term").str.strip_chars().str.extract(r"(\d+)").cast(pl.Int32),
)

inventory = {
    "rows_in_file": RAW_ROWS,
    "footer_rows": footer.height,
    "loans": df.height,
    "columns": len(RAW_COLS),
    "unique_id": df["id"].n_unique(),
    "first_issue_month": str(df["issue_d_dt"].min()),
    "last_issue_month": str(df["issue_d_dt"].max()),
    "last_payment_month_in_data": str(df["last_pymnt_d_dt"].max()),
    "last_credit_pull_month_in_data": str(df["last_credit_pull_d_dt"].max()),
    "file_bytes": (RAW / "accepted_2007_to_2018Q4.csv.gz").stat().st_size,
    "parquet_in_memory_gb": round(df.estimated_size() / 1e9, 2),
    "licences": licences,
}
(OUT / "inventory.json").write_text(json.dumps(inventory, indent=2))
inventory

footer rows: 33
shape: (33, 1)
┌───────────────────────────────────────────────┐
│ id                                            │
│ ---                                           │
│ str                                           │
╞═══════════════════════════════════════════════╡
│ Loans that do not meet the credit policy      │
│ Total amount funded in policy code 2: 1944088 │
│ Total amount funded in policy code 2: 5674470 │
│ Total amount funded in policy code 1: 2087217 │
│ Total amount funded in policy code 1: 1538432 │
│ Total amount funded in policy code 1: 3503840 │
│ Total amount funded in policy code 1: 1443412 │
│ Total amount funded in policy code 1: 1791201 │
│ Total amount funded in policy code 2: 8186622 │
│ Total amount funded in policy code 1: 2063142 │
│ Total amount funded in policy code 2: 6208996 │
│ Total amount funded in policy code 1: 2080429 │
│ Total amount funded in policy code 1: 1741781 │
│ Total amount funded in policy code 1: 1465324 │
│ Total amount fund

{'rows_in_file': 2260701,
 'footer_rows': 33,
 'loans': 2260668,
 'columns': 151,
 'unique_id': 2260668,
 'first_issue_month': '2007-06-01',
 'last_issue_month': '2018-12-01',
 'last_payment_month_in_data': '2019-03-01',
 'last_credit_pull_month_in_data': '2019-04-01',
 'file_bytes': 392582231,
 'parquet_in_memory_gb': 2.72,
 'licences': {'jonchan2003/lending-club-data-dictionary': ['unknown'],
  'wordsforthewise/lending-club': ['CC0-1.0']}}

**Snapshot date.** The latest last-payment month is March 2019 and the latest credit pull
is April 2019, so the file is a picture of every loan as of about end of March 2019. Any
loan still running then has no final outcome in this data.

In [6]:
SNAPSHOT = df["last_pymnt_d_dt"].max()
SNAPSHOT

datetime.date(2019, 3, 1)

## 3. Data dictionary and availability at listing

Rule for `available_at_listing`:

* **yes**: the field appears in Lending Club's Browse Notes dictionary, i.e. an investor
  saw it on the listing.
* **no**: the field describes what happened after the loan was issued (payments,
  recoveries, hardship, settlement, refreshed bureau data) or the result of the funding.
* **doubtful**: not in Browse Notes, but plausibly known at listing; the reason is given.

Column names differ between the two dictionaries (`annual_inc` vs `annualInc`), so they
are matched after removing case and underscores, plus a few explicit synonyms.

In [7]:
def read_dict(path: Path, key: str) -> pl.DataFrame:
    d = pl.read_csv(path, encoding="utf8-lossy", truncate_ragged_lines=True)
    d = d.select(pl.col(key).alias("name"), pl.col("Description").alias("description"))
    return d.filter(pl.col("name").is_not_null()).with_columns(
        pl.col("name").str.replace_all(r"[^A-Za-z0-9_]", ""),
        pl.col("description").str.replace_all("�", "'").str.strip_chars(),
    )


approved = read_dict(RAW / "Lending Club Data Dictionary Approved.csv", "LoanStatNew")
browse = read_dict(RAW / "Lending Club Data Dictionary Notes.csv", "BrowseNotesFile")


def norm(s: str) -> str:
    return re.sub(r"[^a-z0-9]", "", s.lower())


# Same field, different name in the Browse Notes file.
SYNONYMS = {
    "verification_status": "isIncV",
    "verification_status_joint": "verified_status_joint",
    "mo_sin_old_il_acct": "mths_since_oldest_il_open",
    "mths_since_recent_inq": "mthsSinceMostRecentInq",
}
browse_norm = {norm(n) for n in browse["name"]}
desc_map = {norm(r["name"]): r["description"] for r in approved.iter_rows(named=True)}
desc_map.update(
    {
        k: v
        for k, v in {
            norm(r["name"]): r["description"] for r in browse.iter_rows(named=True)
        }.items()
        if k not in desc_map
    }
)
desc_map[norm("verification_status_joint")] = desc_map.get(norm("verified_status_joint"), "")

IDENTIFIERS = {"id", "member_id", "url"}
DOUBTFUL = {
    "issue_d": "Month the loan was funded. Listing happens days before, so the month is "
    "almost always known at listing, but it encodes the vintage (see leakage)",
    "funded_amnt": "Final amount funded. At listing the investor sees the requested amount "
    "(loan_amnt); the two differ in 0.09% of loans. Used as EAD for the targets, not as a feature",
    "mths_since_recent_bc_dlq": "Bureau attribute from the application credit pull, but "
    "not in Browse Notes (only the similar mthsSinceRecentLoanDelinq is)",
}
NOT_AT_LISTING = {
    "funded_amnt_inv": "Amount investors funded: the result of the listing",
    "loan_status": "Outcome",
    "pymnt_plan": "Payment plan after issue",
    "policy_code": "Constant in this file",
}
TEXT = {"emp_title", "desc", "title"}
PROXY = {"addr_state", "zip_code"}
TARGET_SOURCES = {
    "loan_status",
    "total_pymnt",
    "total_rec_prncp",
    "total_rec_int",
    "total_rec_late_fee",
    "recoveries",
    "collection_recovery_fee",
    "last_pymnt_d",
}


def classify(col: str) -> tuple[str, str]:
    n = norm(col)
    in_browse = n in browse_norm or norm(SYNONYMS.get(col, "")) in browse_norm
    if col in DOUBTFUL:
        return "doubtful", DOUBTFUL[col]
    if col in NOT_AT_LISTING:
        return "no", NOT_AT_LISTING[col]
    if in_browse:
        return "yes", "In Browse Notes"
    return "no", "Not in Browse Notes; recorded after issue"


def use(col: str, avail: str) -> str:
    if col in IDENTIFIERS:
        return "identifier"
    if col == "funded_amnt":
        return "target source (EAD)"
    if col in TARGET_SOURCES:
        return "target source"
    if col == "policy_code" or df[col].n_unique() <= 1:
        return "drop (constant)"
    if avail == "no":
        return "drop (leakage)"
    if col in PROXY:
        return "proxy review"
    if col in TEXT:
        return "free text, not used in v1"
    if avail == "doubtful":
        return "decision"
    return "feature candidate"


# First issue year in which a column is filled in for at least 1% of loans: when it starts being reported.
by_year = (
    df.group_by("issue_year")
    .agg([pl.col(c).is_not_null().mean().alias(c) for c in RAW_COLS])
    .sort("issue_year")
)


def first_year(col: str) -> int | None:
    s = by_year.filter(pl.col(col) >= 0.01)["issue_year"]
    return int(s.min()) if s.len() else None


n = df.height
rows = []
for c in RAW_COLS:
    avail, why = classify(c)
    rows.append(
        {
            "column": c,
            "meaning": desc_map.get(norm(c), desc_map.get(norm(SYNONYMS.get(c, "")), "")),
            "type": str(RAW_SCHEMA[c]),
            "pct_missing": round(100 * df[c].null_count() / n, 1),
            "first_year_reported": first_year(c),
            "available_at_listing": avail,
            "reason": why,
            "use": use(c, avail),
        }
    )
dictionary = pl.DataFrame(rows)
dictionary.write_csv(OUT / "data_dictionary.csv")
print(dictionary.group_by("available_at_listing").len())
print(dictionary.group_by("use").len().sort("len", descending=True))

shape: (3, 2)
┌──────────────────────┬─────┐
│ available_at_listing ┆ len │
│ ---                  ┆ --- │
│ str                  ┆ u32 │
╞══════════════════════╪═════╡
│ doubtful             ┆ 3   │
│ yes                  ┆ 107 │
│ no                   ┆ 41  │
└──────────────────────┴─────┘
shape: (9, 2)
┌───────────────────────────┬─────┐
│ use                       ┆ len │
│ ---                       ┆ --- │
│ str                       ┆ u32 │
╞═══════════════════════════╪═════╡
│ feature candidate         ┆ 99  │
│ drop (leakage)            ┆ 32  │
│ target source             ┆ 8   │
│ free text, not used in v1 ┆ 3   │
│ identifier                ┆ 3   │
│ decision                  ┆ 2   │
│ proxy review              ┆ 2   │
│ target source (EAD)       ┆ 1   │
│ drop (constant)           ┆ 1   │
└───────────────────────────┴─────┘


In [8]:
def md_table(d: pl.DataFrame) -> str:
    cols = d.columns
    lines = ["| " + " | ".join(cols) + " |", "|" + "---|" * len(cols)]
    for r in d.iter_rows():
        cells = ["" if v is None else str(v).replace("|", "/").replace("\n", " ") for v in r]
        lines.append("| " + " | ".join(cells) + " |")
    return "\n".join(lines) + "\n"


(OUT / "data_dictionary.md").write_text(md_table(dictionary.drop("reason")))
dictionary.filter(pl.col("available_at_listing") != "yes").select(
    "column", "available_at_listing", "reason", "use"
)

column,available_at_listing,reason,use
str,str,str,str
"""funded_amnt""","""doubtful""","""Final amount funded. At listing the investor sees the reques…","""target source (EAD)"""
"""funded_amnt_inv""","""no""","""Amount investors funded: the result of the listing""","""drop (leakage)"""
"""issue_d""","""doubtful""","""Month the loan was funded. Listing happens days before, so t…","""decision"""
"""loan_status""","""no""","""Outcome""","""target source"""
"""pymnt_plan""","""no""","""Payment plan after issue""","""drop (leakage)"""
"""out_prncp""","""no""","""Not in Browse Notes; recorded after issue""","""drop (leakage)"""
"""out_prncp_inv""","""no""","""Not in Browse Notes; recorded after issue""","""drop (leakage)"""
"""total_pymnt""","""no""","""Not in Browse Notes; recorded after issue""","""target source"""
"""total_pymnt_inv""","""no""","""Not in Browse Notes; recorded after issue""","""drop (leakage)"""


In [9]:
# Columns with no description in either dictionary.
dictionary.filter(pl.col("meaning") == "").select("column")

column
str


## 4. Leakage

### 4.1 Post-issue fields
Every `no` above. The most dangerous ones look like application data but are refreshed
later: `last_fico_range_*` and `last_credit_pull_d`. Their gap with the application FICO
shows how much they move after issue, and how strongly that movement tracks the outcome.

In [10]:
fico = df.filter(pl.col("loan_status").is_in(["Fully Paid", "Charged Off"])).select(
    "loan_status",
    (pl.col("last_fico_range_low") - pl.col("fico_range_low")).alias("fico_change"),
)
fico.group_by("loan_status").agg(
    pl.col("fico_change").median().alias("median_change"),
    pl.col("fico_change").quantile(0.1).alias("p10"),
    pl.col("fico_change").quantile(0.9).alias("p90"),
)

loan_status,median_change,p10,p90
str,f64,f64,f64
"""Fully Paid""",5.0,-70.0,70.0
"""Charged Off""",-130.0,-660.0,-45.0


### 4.2 Fields that encode the vintage
Columns that only start being reported in a given year: their missingness alone tells
the model the loan's vintage. Shown as the share of non-missing values by issue year.

In [11]:
groups = {
    "mths_since_recent_inq (bureau, 2012+)": "mths_since_recent_inq",
    "tot_cur_bal (bureau, ~2012+)": "tot_cur_bal",
    "open_acc_6m (bureau, 2015-12+)": "open_acc_6m",
    "sec_app_fico_range_low (joint, 2017+)": "sec_app_fico_range_low",
    "desc (free text, until 2014)": "desc",
    "emp_title": "emp_title",
}
vintage = (
    df.group_by("issue_year")
    .agg(
        pl.len().alias("loans"),
        *[pl.col(c).is_not_null().mean().round(3).alias(k) for k, c in groups.items()],
        (pl.col("application_type") == "Joint App").mean().round(3).alias("joint_share"),
    )
    .sort("issue_year")
)
vintage.write_csv(OUT / "missingness_by_year.csv")
vintage

issue_year,loans,"mths_since_recent_inq (bureau, 2012+)","tot_cur_bal (bureau, ~2012+)","open_acc_6m (bureau, 2015-12+)","sec_app_fico_range_low (joint, 2017+)","desc (free text, until 2014)",emp_title,joint_share
i32,u32,f64,f64,f64,f64,f64,f64,f64
2007,603,0.0,0.0,0.0,0.0,0.975,0.91,0.0
2008,2393,0.0,0.0,0.0,0.0,1.0,0.959,0.0
2009,5281,0.0,0.0,0.0,0.0,0.97,0.944,0.0
2010,12537,0.0,0.0,0.0,0.0,0.671,0.941,0.0
2011,21721,0.0,0.0,0.0,0.0,0.586,0.934,0.0
2012,53367,0.751,0.48,0.0,0.0,0.614,0.941,0.0
2013,134814,0.892,1.0,0.0,0.0,0.361,0.936,0.0
2014,235629,0.908,1.0,0.0,0.0,0.065,0.944,0.0
2015,421095,0.894,1.0,0.051,0.0,0.0,0.943,0.001


In [12]:
# Month the 'open_acc_6m' family appears.
(
    df.group_by("issue_d_dt")
    .agg(pl.col("open_acc_6m").is_not_null().mean().alias("share"))
    .filter(pl.col("share") > 0.5)
    .sort("issue_d_dt")
    .head(1)
)

issue_d_dt,share
date,f64
2016-01-01,0.998084


In [13]:
# From here on only these columns are needed. Dropping the rest keeps the notebook within
# about 10 GB of memory (the first version ran out of memory at 14 GB; see data_audit.md, E-5).
KEEP = [
    "id",
    "member_id",
    "url",
    "loan_amnt",
    "funded_amnt",
    "term",
    "term_m",
    "int_rate",
    "grade",
    "sub_grade",
    "emp_length",
    "home_ownership",
    "annual_inc",
    "issue_d_dt",
    "issue_year",
    "loan_status",
    "addr_state",
    "zip_code",
    "dti",
    "earliest_cr_line",
    "earliest_cr_line_dt",
    "fico_range_low",
    "revol_util",
    "total_pymnt",
    "total_rec_prncp",
    "total_rec_int",
    "total_rec_late_fee",
    "recoveries",
    "collection_recovery_fee",
    "last_pymnt_d_dt",
    "application_type",
]
df = df.select(KEEP)
del fico  # by_year is kept: first_year() reads it

## 5. Labels and their delay

### 5.1 Status mapping

In [14]:
status = (
    df.group_by("loan_status")
    .agg(
        pl.len().alias("loans"),
        pl.col("issue_d_dt").min().alias("first_issue"),
        pl.col("issue_d_dt").max().alias("last_issue"),
    )
    .sort("loans", descending=True)
)
status.write_csv(OUT / "loan_status.csv")
status

loan_status,loans,first_issue,last_issue
str,u32,date,date
"""Fully Paid""",1076751,2007-06-01,2018-12-01
"""Current""",878317,2013-10-01,2018-12-01
"""Charged Off""",268559,2007-08-01,2018-12-01
"""Late (31-120 days)""",21467,2013-12-01,2018-12-01
"""In Grace Period""",8436,2013-12-01,2018-12-01
"""Late (16-30 days)""",4349,2014-01-01,2018-12-01
"""Does not meet the credit policy. Status:Fully Paid""",1988,2007-06-01,2010-12-01
"""Does not meet the credit policy. Status:Charged Off""",761,2007-06-01,2010-12-01
"""Default""",40,2014-08-01,2018-07-01


Mapping decided by Juan (D-013, 2026-11-21):

| Status | Label | Why |
|---|---|---|
| Fully Paid | 0 | Final |
| Charged Off, Default | 1 | Final, or 121+ days late and about to be charged off |
| Does not meet the credit policy. Status: Fully Paid / Charged Off | removed | Approved under an older, looser policy (2007 to 2010); an API user expects scores under the current policy |
| Current, In Grace Period, Late (16-30), Late (31-120) | none | Outcome not final at the snapshot |

From here on, every figure excludes the removed loans.

In [15]:
OLD_POLICY = pl.col("loan_status").str.starts_with("Does not meet the credit policy")
print("removed (older credit policy):", df.filter(OLD_POLICY).height)
df = df.filter(~OLD_POLICY)
FINAL_BAD = ["Charged Off", "Default"]
FINAL_GOOD = ["Fully Paid"]
df = df.with_columns(
    resolved=pl.col("loan_status").is_in(FINAL_BAD + FINAL_GOOD),
    bad=pl.when(pl.col("loan_status").is_in(FINAL_BAD))
    .then(1)
    .when(pl.col("loan_status").is_in(FINAL_GOOD))
    .then(0)
    .otherwise(None),
)
print(df["loan_status"].unique().to_list())

removed (older credit policy): 2749
['In Grace Period', 'Charged Off', 'Late (16-30 days)', 'Late (31-120 days)', 'Fully Paid', 'Default', 'Current']


### 5.2 Is there a default date?

There is **no charge-off date column**. What exists:

* `last_pymnt_d`: month of the last payment received. For a charged-off loan, the first
  missed payment is the month after.
* Lending Club's policy: a loan is charged off when it reaches about 150 days past due
  (so about 5 months after the first missed payment).

So the default month can be **estimated** as `last_pymnt_d + 1 month` (first missed payment)
and the month the outcome becomes known as `last_pymnt_d + 5 months` (charge-off). The checks
below test how usable this is.

In [16]:
co = df.filter(pl.col("bad") == 1)
print("bad loans:", co.height)
print("bad loans with no last_pymnt_d (never paid):", co["last_pymnt_d_dt"].null_count())
co_m = co.with_columns(
    months_to_last_pay=(
        (pl.col("last_pymnt_d_dt").dt.year() - pl.col("issue_d_dt").dt.year()) * 12
        + pl.col("last_pymnt_d_dt").dt.month()
        - pl.col("issue_d_dt").dt.month()
    )
)
print(co_m["months_to_last_pay"].describe())

# Charged-off loans whose last payment is less than 5 months before the snapshot would not
# have reached charge-off yet if the rule held: a consistency check of the rule.
print(
    "bad loans with last payment within 4 months of the snapshot:",
    co_m.filter(pl.col("last_pymnt_d_dt") > pl.lit(SNAPSHOT).dt.offset_by("-4mo")).height,
)

bad loans: 268599
bad loans with no last_pymnt_d (never paid): 2313
shape: (9, 2)
┌────────────┬───────────┐
│ statistic  ┆ value     │
│ ---        ┆ ---       │
│ str        ┆ f64       │
╞════════════╪═══════════╡
│ count      ┆ 266286.0  │
│ null_count ┆ 2313.0    │
│ mean       ┆ 16.058535 │
│ std        ┆ 9.840907  │
│ min        ┆ 0.0       │
│ 25%        ┆ 8.0       │
│ 50%        ┆ 14.0      │
│ 75%        ┆ 22.0      │
│ max        ┆ 66.0      │
└────────────┴───────────┘
bad loans with last payment within 4 months of the snapshot: 4015


In [17]:
# Early-warning metric feasibility: share of bad loans whose first missed payment falls in
# the first 12 months, by term.
co_m.group_by("term_m").agg(
    pl.len().alias("bad_loans"),
    (pl.col("months_to_last_pay").fill_null(0) + 1 <= 12)
    .mean()
    .round(3)
    .alias("first_missed_within_12m"),
).sort("term_m")

term_m,bad_loans,first_missed_within_12m
i32,u32,f64
36,163277,0.41
60,105322,0.359


### 5.3 Delay from issue to a final label

Resolution month: payoff month for good loans (`last_pymnt_d`), estimated charge-off month
for bad loans (`last_pymnt_d + 5`, or `issue + 5` if never paid), capped at the snapshot
because every charged-off loan in the file was charged off by then. The 4,015 bad loans with
a last payment less than 5 months before the snapshot show that the 5-month lag is a typical
value, not a fixed rule (bankruptcy, for example, is charged off sooner).

In [18]:
def add_months(date_col: str, k: int | str) -> pl.Expr:
    k_expr = pl.lit(k) if isinstance(k, int) else pl.col(k)
    return pl.col(date_col).dt.offset_by(pl.format("{}mo", k_expr))


df = df.with_columns(
    resolution_dt=pl.when(pl.col("bad") == 1)
    .then(
        pl.min_horizontal(
            pl.coalesce(add_months("last_pymnt_d_dt", 5), add_months("issue_d_dt", 5)),
            pl.lit(SNAPSHOT),
        )
    )
    .when(pl.col("bad") == 0)
    .then(pl.col("last_pymnt_d_dt"))
    .otherwise(None)
)
df = df.with_columns(
    label_delay_m=(
        (pl.col("resolution_dt").dt.year() - pl.col("issue_d_dt").dt.year()) * 12
        + pl.col("resolution_dt").dt.month()
        - pl.col("issue_d_dt").dt.month()
    )
)
delay = (
    df.filter(pl.col("resolved"))
    .group_by("term_m", "bad")
    .agg(
        pl.len().alias("loans"),
        *[
            pl.col("label_delay_m").quantile(q).alias(f"p{int(q * 100)}")
            for q in (0.1, 0.25, 0.5, 0.75, 0.9)
        ],
        (pl.col("label_delay_m") >= pl.col("term_m"))
        .mean()
        .round(3)
        .alias("share_at_or_after_term"),
    )
    .sort("term_m", "bad")
)
delay.write_csv(OUT / "label_delay.csv")
delay

term_m,bad,loans,p10,p25,p50,p75,p90,share_at_or_after_term
i32,i32,u32,f64,f64,f64,f64,f64,f64
36,0,857491,5.0,12.0,22.0,35.0,36.0,0.245
36,1,163277,9.0,13.0,19.0,26.0,32.0,0.049
60,0,219260,4.0,9.0,19.0,33.0,48.0,0.048
60,1,105322,10.0,14.0,20.0,30.0,39.0,0.003


### 5.4 Share of loans with a final outcome at the snapshot, by vintage and term

And the default rate among those resolved. When only part of a vintage is resolved, the
resolved part is biased: early defaults and early prepayments end first.

In [19]:
res = (
    df.group_by("issue_year", "term_m")
    .agg(
        pl.len().alias("loans"),
        pl.col("resolved").mean().round(3).alias("share_resolved"),
        pl.col("bad").mean().round(3).alias("bad_rate_among_resolved"),
    )
    .sort("issue_year", "term_m")
)
res.write_csv(OUT / "resolved_by_vintage.csv")
res.pivot(
    on="term_m", index="issue_year", values=["loans", "share_resolved", "bad_rate_among_resolved"]
)

issue_year,loans_36,loans_60,share_resolved_36,share_resolved_60,bad_rate_among_resolved_36,bad_rate_among_resolved_60
i32,u32,u32,f64,f64,f64,f64
2007,251,null,1.0,null,0.179,null
2008,1562,null,1.0,null,0.158,null
2009,4716,null,1.0,null,0.126,null
2010,8466,3070,1.0,1.0,0.099,0.21
2011,14101,7620,1.0,1.0,0.106,0.236
2012,43470,9897,1.0,1.0,0.136,0.277
2013,100422,34392,1.0,1.0,0.123,0.251
2014,162570,73059,1.0,0.829,0.137,0.311
2015,283173,137922,0.999,0.671,0.149,0.364


### 5.5 Realised return and LGD: does the data support them?

* `total_pymnt` should equal principal + interest + late fees + recoveries.
* Net realised return of a loan = `(total_pymnt - collection_recovery_fee - funded_amnt) / funded_amnt`.
  Lending Club's 1% investor service fee is not in the data (documented limitation).
* LGD = `(EAD - total_rec_prncp - (recoveries - collection_recovery_fee)) / EAD` with EAD = `funded_amnt`.

In [20]:
chk = df.select(
    (
        pl.col("total_pymnt")
        - (
            pl.col("total_rec_prncp")
            + pl.col("total_rec_int")
            + pl.col("total_rec_late_fee")
            + pl.col("recoveries")
        )
    )
    .abs()
    .alias("gap")
)
print(
    "max |total_pymnt - components|:",
    chk["gap"].max(),
    "| share above 1 dollar:",
    (chk["gap"] > 1).mean(),
)

fin = df.filter(pl.col("resolved")).with_columns(
    net_return=(pl.col("total_pymnt") - pl.col("collection_recovery_fee") - pl.col("funded_amnt"))
    / pl.col("funded_amnt"),
    lgd=(
        pl.col("funded_amnt")
        - pl.col("total_rec_prncp")
        - (pl.col("recoveries") - pl.col("collection_recovery_fee"))
    )
    / pl.col("funded_amnt"),
)
fin.group_by("bad").agg(
    pl.len(),
    pl.col("net_return").median().round(3).alias("median_net_return"),
    pl.col("net_return").mean().round(3).alias("mean_net_return"),
)

max |total_pymnt - components|: 0.010000000000218279 | share above 1 dollar: 0.0


bad,len,median_net_return,mean_net_return
i32,u32,f64,f64
0,1076751,0.136,0.157
1,268599,-0.508,-0.471


In [21]:
lgd = fin.filter(pl.col("bad") == 1)
print(lgd["lgd"].describe())
print("LGD outside [0, 1]:", lgd.filter((pl.col("lgd") < 0) | (pl.col("lgd") > 1)).height)
# Recoveries take time: share of bad loans with zero recoveries by charge-off year.
lgd_year = (
    lgd.group_by(pl.col("resolution_dt").dt.year().alias("charge_off_year"))
    .agg(
        pl.len().alias("bad_loans"),
        (pl.col("recoveries") == 0).mean().round(3).alias("share_zero_recoveries"),
        pl.col("lgd").mean().round(3).alias("mean_lgd"),
    )
    .sort("charge_off_year")
)
lgd_year.write_csv(OUT / "lgd_by_chargeoff_year.csv")
lgd_year

shape: (9, 2)
┌────────────┬──────────┐
│ statistic  ┆ value    │
│ ---        ┆ ---      │
│ str        ┆ f64      │
╞════════════╪══════════╡
│ count      ┆ 268599.0 │
│ null_count ┆ 0.0      │
│ mean       ┆ 0.634423 │
│ std        ┆ 0.214953 │
│ min        ┆ -1.1703  │
│ 25%        ┆ 0.497487 │
│ 50%        ┆ 0.676455 │
│ 75%        ┆ 0.801354 │
│ max        ┆ 1.0      │
└────────────┴──────────┘
LGD outside [0, 1]: 86


charge_off_year,bad_loans,share_zero_recoveries,mean_lgd
i32,u32,f64,f64
2008,13,0.077,0.766
2009,139,0.05,0.747
2010,397,0.01,0.714
2011,766,0.009,0.728
2012,1756,0.008,0.72
2013,4432,0.011,0.732
2014,10722,0.078,0.674
2015,23840,0.173,0.654
2016,47607,0.218,0.665


How long recoveries take to arrive: net recovery (share of the funded amount) by months
between the estimated charge-off and the snapshot. Basis for D-017 (LGD trained only on loans
charged off at least 12 months earlier). Age and calendar era cannot be separated with a single
snapshot: the oldest charge-offs are also from another collection era.

In [22]:
rec_age = (
    lgd.with_columns(
        months_since_charge_off=(
            (SNAPSHOT.year - pl.col("resolution_dt").dt.year()) * 12
            + SNAPSHOT.month
            - pl.col("resolution_dt").dt.month()
        ),
        net_recovery_rate=(pl.col("recoveries") - pl.col("collection_recovery_fee"))
        / pl.col("funded_amnt"),
    )
    .with_columns(age_bucket=(pl.col("months_since_charge_off") // 6 * 6).clip(0, 60))
    .group_by("age_bucket")
    .agg(
        pl.len().alias("bad_loans"),
        (pl.col("recoveries") == 0).mean().round(3).alias("share_zero_recoveries"),
        pl.col("net_recovery_rate").mean().round(4).alias("mean_net_recovery_rate"),
        pl.col("lgd").mean().round(3).alias("mean_lgd"),
    )
    .sort("age_bucket")
)
rec_age.write_csv(OUT / "recovery_by_months_since_chargeoff.csv")
rec_age

age_bucket,bad_loans,share_zero_recoveries,mean_net_recovery_rate,mean_lgd
i32,u32,f64,f64,f64
0,48129,0.634,0.0266,0.619
6,37150,0.341,0.0603,0.604
12,40239,0.289,0.0708,0.604
18,35054,0.267,0.0714,0.627
24,34537,0.228,0.0663,0.66
30,21697,0.221,0.0739,0.665
36,17940,0.197,0.0853,0.652
42,10824,0.184,0.084,0.656
48,8474,0.126,0.0832,0.666


## 6. Data quality

In [23]:
quality = {
    "duplicate_id": df.height - df["id"].n_unique(),
    "member_id_all_null": df["member_id"].null_count() == df.height,
    "url_id_mismatch": int((df["url"].str.extract(r"loan_id=(\d+)") != df["id"]).sum()),
    "term_values": sorted(df["term"].unique().to_list()),
    "dti_negative": df.filter(pl.col("dti") < 0).height,
    "dti_999": df.filter(pl.col("dti") >= 999).height,
    "dti_above_100": df.filter(pl.col("dti") > 100).height,
    "dti_null": df["dti"].null_count(),
    "annual_inc_zero": df.filter(pl.col("annual_inc") == 0).height,
    "annual_inc_above_10M": df.filter(pl.col("annual_inc") > 10_000_000).height,
    "revol_util_above_100": df.filter(pl.col("revol_util") > 100).height,
    "earliest_cr_line_unparsable": df.filter(
        pl.col("earliest_cr_line").is_not_null() & pl.col("earliest_cr_line_dt").is_null()
    ).height,
    "earliest_cr_line_null": df["earliest_cr_line"].null_count(),
    "earliest_cr_line_after_issue": df.filter(
        pl.col("earliest_cr_line_dt") > pl.col("issue_d_dt")
    ).height,
    "funded_lt_requested": df.filter(pl.col("funded_amnt") < pl.col("loan_amnt")).height,
    "home_ownership_values": dict(sorted(df["home_ownership"].value_counts().iter_rows())),
    "emp_length_null": df["emp_length"].null_count(),
    "int_rate_range": [df["int_rate"].min(), df["int_rate"].max()],
    "fico_range_low_min": df["fico_range_low"].min(),
    "loan_amnt_range": [df["loan_amnt"].min(), df["loan_amnt"].max()],
}
(OUT / "quality.json").write_text(json.dumps(quality, indent=2, default=str))
quality

{'duplicate_id': 0,
 'member_id_all_null': True,
 'url_id_mismatch': 0,
 'term_values': [' 36 months', ' 60 months'],
 'dti_negative': 2,
 'dti_999': 135,
 'dti_above_100': 2561,
 'dti_null': 1711,
 'annual_inc_zero': 1667,
 'annual_inc_above_10M': 3,
 'revol_util_above_100': 7316,
 'earliest_cr_line_unparsable': 0,
 'earliest_cr_line_null': 0,
 'earliest_cr_line_after_issue': 0,
 'funded_lt_requested': 1924,
 'home_ownership_values': {'ANY': 996,
  'MORTGAGE': 1110194,
  'NONE': 49,
  'OTHER': 144,
  'OWN': 252870,
  'RENT': 893666},
 'emp_length_null': 146873,
 'int_rate_range': [5.31, 30.99],
 'fico_range_low_min': 625.0,
 'loan_amnt_range': [500.0, 40000.0]}

In [24]:
# Does grade mean the same thing over time? Mean interest rate by grade and year.
df.group_by("issue_year", "grade").agg(pl.col("int_rate").mean().round(1)).pivot(
    on="grade", index="issue_year", values="int_rate", sort_columns=True
).sort("issue_year")

issue_year,A,B,C,D,E,F,G
i32,f64,f64,f64,f64,f64,f64,f64
2007,7.7,9.4,10.8,12.4,14.2,15.5,null
2008,8.4,10.4,11.8,13.3,15.0,16.5,18.4
2009,8.6,11.8,13.3,14.9,16.5,18.3,20.0
2010,7.1,10.7,13.5,15.3,16.9,18.7,20.6
2011,7.1,11.1,13.9,16.4,18.4,20.4,22.0
2012,7.6,12.1,15.3,18.4,21.1,23.1,24.5
2013,7.7,11.8,15.5,18.7,21.7,24.0,25.5
2014,7.5,11.2,14.1,17.1,20.4,24.2,25.9
2015,6.9,10.0,13.3,16.7,19.3,23.6,26.8


## 7. Simulation over time

### 7.1 Loans listed per month

In [25]:
monthly = df.group_by("issue_d_dt", "term_m").len().sort("issue_d_dt", "term_m")
monthly.write_csv(OUT / "loans_per_month.csv")
wide = monthly.pivot(on="term_m", index="issue_d_dt", values="len").sort("issue_d_dt").fill_null(0)
fig, ax = plt.subplots(figsize=(9, 3.5))
ax.stackplot(
    wide["issue_d_dt"].to_list(),
    wide["36"].to_list(),
    wide["60"].to_list(),
    labels=["36 months", "60 months"],
)
ax.set_ylabel("loans issued per month")
ax.legend(loc="upper left")
ax.set_title("Lending Club loans issued per month")
fig.tight_layout()
fig.savefig(OUT / "loans_per_month.png", dpi=120)
df.group_by("issue_year").len().sort("issue_year").transpose(include_header=True)

column,column_0,column_1,column_2,column_3,column_4,column_5,column_6,column_7,column_8,column_9,column_10,column_11
str,i64,i64,i64,i64,i64,i64,i64,i64,i64,i64,i64,i64
"""issue_year""",2007,2008,2009,2010,2011,2012,2013,2014,2015,2016,2017,2018
"""len""",251,1562,4716,11536,21721,53367,134814,235629,421095,434407,443579,495242


### 7.2 Labels available for the first training, by start date

At a simulated date T the system may only use labels already known at T. Two rules:

* **Known by T**: the loan was resolved (estimated resolution month) on or before T.
  Biased for recent vintages: early defaults and early prepayments are over-represented.
* **Matured cohorts**: only loans issued at least `term` months before T, all of whose
  outcomes are known (up to the 5-month charge-off lag). Unbiased, but older.

In [26]:
from datetime import date

cands = [date(y, 1, 1) for y in range(2010, 2019)]
rows = []
for T in cands:
    known = df.filter(pl.col("resolution_dt") <= T)
    matured = df.filter(
        pl.col("resolved")
        & (pl.col("issue_d_dt").dt.offset_by(pl.format("{}mo", pl.col("term_m") + 5)) <= T)
    )
    rows.append(
        {
            "T": T,
            "known_by_T": known.height,
            "known_by_T_bad_rate": round(known["bad"].mean(), 3),
            "matured_cohorts": matured.height,
            "matured_bad_rate": round(matured["bad"].mean(), 3) if matured.height else None,
            "loans_listed_after_T": df.filter(pl.col("issue_d_dt") >= T).height,
        }
    )
first_train = pl.DataFrame(rows)
first_train.write_csv(OUT / "labels_at_start_date.csv")
first_train

T,known_by_T,known_by_T_bad_rate,matured_cohorts,matured_bad_rate,loans_listed_after_T
date,i64,f64,i64,f64,i64
2010-01-01,640,0.27,0,null,2251390
2011-01-01,2571,0.231,64,0.109,2239854
2012-01-01,7559,0.188,1278,0.163,2218133
2013-01-01,18951,0.175,4335,0.138,2164766
2014-01-01,45852,0.178,12078,0.121,2029952
2015-01-01,113208,0.174,23358,0.11,1794323
2016-01-01,260452,0.173,53682,0.127,1373228
2017-01-01,502116,0.19,139593,0.131,938821
2018-01-01,853582,0.199,294217,0.134,495242


### 7.3 Loans that could be evaluated on their final outcome
The business metric (D-007) needs the final outcome. Share of each listing year with a final
outcome at the snapshot is in 5.4. Cumulative view for a simulation that ends at year E:

In [27]:
eval_ok = (
    df.group_by("issue_year")
    .agg(
        pl.len().alias("loans"),
        pl.col("resolved").mean().round(3).alias("share_resolved_all_terms"),
        pl.col("resolved")
        .filter(pl.col("term_m") == 36)
        .mean()
        .round(3)
        .alias("share_resolved_36m"),
    )
    .sort("issue_year")
)
eval_ok

issue_year,loans,share_resolved_all_terms,share_resolved_36m
i32,u32,f64,f64
2007,251,1.0,1.0
2008,1562,1.0,1.0
2009,4716,1.0,1.0
2010,11536,1.0,1.0
2011,21721,1.0,1.0
2012,53367,1.0,1.0
2013,134814,1.0,1.0
2014,235629,0.947,1.0
2015,421095,0.892,0.999


## 8. Indirect discrimination: `addr_state` and zip code

Neither field is a protected attribute, but both are classic proxies: in the US, where
someone lives correlates with race and ethnicity, and using it in credit decisions is the
pattern known as redlining. The question for the project is: **how much predictive value
do they add beyond the grade, compared with the risk they carry?**

Without demographic data we cannot measure the proxy effect directly, so this section
measures (a) how concentrated the fields are and (b) their incremental value over `grade`
on an out-of-time test, with matured 36-month loans only.

In [28]:
print("states:", df["addr_state"].n_unique(), "| zip3 prefixes:", df["zip_code"].n_unique())
zip_counts = df.group_by("zip_code").len()
print("zip3 with fewer than 100 loans:", zip_counts.filter(pl.col("len") < 100).height)
print("zip3 with fewer than 1000 loans:", zip_counts.filter(pl.col("len") < 1000).height)

m36 = df.filter((pl.col("term_m") == 36) & pl.col("resolved") & (pl.col("issue_year") <= 2015))
state_rates = (
    m36.group_by("addr_state")
    .agg(pl.len().alias("loans"), pl.col("bad").mean().round(3).alias("bad_rate"))
    .filter(pl.col("loans") >= 1000)
    .sort("bad_rate", "addr_state")
)
print(
    "bad rate across states with 1000+ loans: min",
    state_rates["bad_rate"].min(),
    "max",
    state_rates["bad_rate"].max(),
)
state_rates.write_csv(OUT / "bad_rate_by_state.csv")
pl.concat([state_rates.head(5), state_rates.tail(5)])

states: 51 | zip3 prefixes: 956
zip3 with fewer than 100 loans: 111
zip3 with fewer than 1000 loans: 447
bad rate across states with 1000+ loans: min 0.092 max 0.187


addr_state,loans,bad_rate
str,u32,f64
"""NH""",2952,0.092
"""DC""",1822,0.095
"""VT""",1237,0.1
"""WV""",2746,0.104
"""KS""",5351,0.109
"""AL""",7296,0.163
"""OK""",5474,0.168
"""AR""",4552,0.17
"""NV""",9169,0.174


In [29]:
# Incremental value: logistic regression on grade, vs grade + state, vs grade + zip3.
# Train: 36-month loans issued 2012-2014; test: issued 2015 (both almost fully resolved).
train = m36.filter(pl.col("issue_year").is_between(2012, 2014))
test = m36.filter(pl.col("issue_year") == 2015)


def fit_eval(cols: list[str]) -> dict:
    enc = OneHotEncoder(handle_unknown="ignore", min_frequency=200)
    Xtr = enc.fit_transform(train.select(cols).to_pandas())
    Xte = enc.transform(test.select(cols).to_pandas())
    clf = LogisticRegression(max_iter=2000, C=1.0)
    clf.fit(Xtr, train["bad"].to_numpy())
    p = clf.predict_proba(Xte)[:, 1]
    y = test["bad"].to_numpy()
    return {
        "features": "+".join(cols),
        "test_auc": round(roc_auc_score(y, p), 4),
        "test_log_loss": round(log_loss(y, p), 5),
    }


proxy_value = pl.DataFrame(
    [
        fit_eval(["sub_grade"]),
        fit_eval(["sub_grade", "addr_state"]),
        fit_eval(["sub_grade", "zip_code"]),
    ]
)
print("train", train.height, "test", test.height)
proxy_value.write_csv(OUT / "proxy_incremental_value.csv")
proxy_value

train 306462 test 283026


features,test_auc,test_log_loss
str,f64,f64
"""sub_grade""",0.6786,0.39726
"""sub_grade+addr_state""",0.6808,0.39657
"""sub_grade+zip_code""",0.6783,0.39707


Bootstrap of the AUC gain, to compare it with noise (200 resamples of the test set).

In [30]:
rng = np.random.default_rng(0)


def scores(cols):
    enc = OneHotEncoder(handle_unknown="ignore", min_frequency=200)
    Xtr = enc.fit_transform(train.select(cols).to_pandas())
    clf = LogisticRegression(max_iter=2000).fit(Xtr, train["bad"].to_numpy())
    return clf.predict_proba(enc.transform(test.select(cols).to_pandas()))[:, 1]


y = test["bad"].to_numpy()
p_g, p_s, p_z = (
    scores(["sub_grade"]),
    scores(["sub_grade", "addr_state"]),
    scores(["sub_grade", "zip_code"]),
)
gains_s, gains_z = [], []
for _ in range(200):
    i = rng.integers(0, len(y), len(y))
    base = roc_auc_score(y[i], p_g[i])
    gains_s.append(roc_auc_score(y[i], p_s[i]) - base)
    gains_z.append(roc_auc_score(y[i], p_z[i]) - base)
boot = {
    "auc_gain_state_mean": round(float(np.mean(gains_s)), 4),
    "auc_gain_state_95ci": [
        round(float(np.percentile(gains_s, 2.5)), 4),
        round(float(np.percentile(gains_s, 97.5)), 4),
    ],
    "auc_gain_zip3_mean": round(float(np.mean(gains_z)), 4),
    "auc_gain_zip3_95ci": [
        round(float(np.percentile(gains_z, 2.5)), 4),
        round(float(np.percentile(gains_z, 97.5)), 4),
    ],
}
(OUT / "proxy_bootstrap.json").write_text(json.dumps(boot, indent=2))
boot

{'auc_gain_state_mean': 0.0021,
 'auc_gain_state_95ci': [0.0016, 0.0028],
 'auc_gain_zip3_mean': -0.0003,
 'auc_gain_zip3_95ci': [-0.0011, 0.0005]}

## 9. Environment used for this run

In [31]:
import importlib.metadata as md

{p: md.version(p) for p in ["polars", "pyarrow", "scikit-learn", "numpy", "matplotlib", "kaggle"]}

{'polars': '2.1.0',
 'pyarrow': '26.0.1',
 'scikit-learn': '1.9.1',
 'numpy': '2.5.5',
 'matplotlib': '3.11.2',
 'kaggle': '2.2.5'}